# SigLIP2 Lower — ONNX Conversion & Validation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
"""
SigLIP2 Sub-Classifier — PT → ONNX Conversion & Validation
============================================================
Converts both upper and lower SigLIP2 .pt checkpoints to .onnx
with comprehensive validation before TRT conversion on Jetson.

Validations performed:
  1. Checkpoint integrity (keys, shapes, class names)
  2. PyTorch inference sanity check
  3. ONNX export with softmax baked in
  4. ONNX model structure verification
  5. ONNX Runtime numerical validation (PyTorch vs ONNX output match)
  6. Output tensor checks (softmax sum=1, correct shape, correct name)
  7. Batch of real images validation (if test images available)
  8. ONNX simplification (onnxsim)
  9. Final simplified ONNX vs PyTorch validation
  10. Production config JSON export

Run in Colab:
  Cell 1: !pip install --upgrade transformers onnx onnxruntime-gpu onnxsim --quiet
  Cell 2: Run this script
"""
!pip install onnx onnxruntime onnxscript
import os, json, sys, warnings, time
from pathlib import Path
import numpy as np
from PIL import Image

import torch
import torch.nn.functional as F
from torch import nn

import onnx
import onnxruntime as ort

warnings.filterwarnings("ignore")


# ============================================================
# CONFIG — UPDATE THESE PATHS
# ============================================================

MODELS = {
    "upper": {
        "pt_path": "/content/drive/Shareddrives/Garment Type/Complete_dataset/Siglip2_models/siglip2_upper_sub/best_siglip2_upper.pt",
        "onnx_dir": "/content/drive/Shareddrives/Garment Type/Complete_dataset/Siglip2_models/siglip2_upper_sub",
        "onnx_name": "siglip2_upper",
        "thresh_path": "/content/drive/Shareddrives/Garment Type/Complete_dataset/Siglip2_models/siglip2_upper_sub/upper_siglip2_thresholds.json",
        # Optional: path to a folder with a few test images for real-image validation
        "test_images_dir": None,  # e.g., "/content/drive/.../classified_images_13_14_test/shirt"
    },
    "lower": {
        "pt_path": "/content/drive/Shareddrives/Garment Type/Complete_dataset/Siglip2_models/siglip2_lower_sub/best_siglip2_lower.pt",
        "onnx_dir": "/content/drive/Shareddrives/Garment Type/Complete_dataset/Siglip2_models/siglip2_lower_sub",
        "onnx_name": "siglip2_lower",
        "thresh_path": "/content/drive/Shareddrives/Garment Type/Complete_dataset/Siglip2_models/siglip2_lower_sub/lower_siglip2_thresholds.json",
        "test_images_dir": None,
    },
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SIGLIP2_MODEL_ID = "google/siglip2-so400m-patch14-384"
IMG_SIZE = 384
HIDDEN_SIZE = 1152
NORM_MEAN = (0.5, 0.5, 0.5)
NORM_STD = (0.5, 0.5, 0.5)

# Tolerance for numerical comparison
ATOL = 1e-4   # absolute tolerance
RTOL = 1e-3   # relative tolerance


# ============================================================
# MODEL DEFINITIONS
# ============================================================
class SigLIP2Classifier(nn.Module):
    """Must match training definition exactly."""
    def __init__(self, model_id, num_classes, dropout_rate=0.0):
        super().__init__()
        from transformers import AutoModel
        full_model = AutoModel.from_pretrained(model_id, ignore_mismatched_sizes=True)
        self.vision = full_model.vision_model
        del full_model

        self.classifier = nn.Sequential(
            nn.LayerNorm(HIDDEN_SIZE),
            nn.Dropout(dropout_rate),
            nn.Linear(HIDDEN_SIZE, num_classes),
        )

    def forward(self, pixel_values):
        outputs = self.vision(pixel_values=pixel_values)
        pooled = outputs.pooler_output
        return self.classifier(pooled)


class SigLIP2ForExport(nn.Module):
    """Wrapper: bakes softmax into output for TRT deployment.
    Output tensor is 'probabilities' — do NOT apply softmax again."""
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, pixel_values):
        logits = self.model(pixel_values)
        return F.softmax(logits, dim=1)


# ============================================================
# VALIDATION HELPERS
# ============================================================
def passed(msg):
    print(f"  ✅ {msg}")

def failed(msg):
    print(f"  ❌ {msg}")
    return False

def warn(msg):
    print(f"  ⚠️  {msg}")


def preprocess_image(img_path):
    """Preprocess a single image matching production pipeline."""
    img = Image.open(img_path).convert("RGB")
    img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    arr = np.array(img, dtype=np.float32) / 255.0
    arr = (arr - np.array(NORM_MEAN)) / np.array(NORM_STD)
    arr = arr.transpose(2, 0, 1)  # HWC → CHW
    return arr[np.newaxis, ...]   # add batch dim


# ============================================================
# MAIN CONVERSION + VALIDATION
# ============================================================
def convert_and_validate(branch, config):
    print(f"\n{'#'*70}")
    print(f"  CONVERTING: SigLIP2 {branch.upper()} Sub-Classifier")
    print(f"{'#'*70}")

    pt_path = Path(config["pt_path"])
    onnx_dir = Path(config["onnx_dir"])
    onnx_name = config["onnx_name"]
    all_passed = True

    # ──────────────────────────────────────────────
    # 1. CHECKPOINT INTEGRITY
    # ──────────────────────────────────────────────
    print(f"\n{'─'*50}")
    print(f"  [1/10] Checkpoint Integrity")
    print(f"{'─'*50}")

    if not pt_path.exists():
        return failed(f"Checkpoint not found: {pt_path}")

    ckpt = torch.load(pt_path, map_location="cpu", weights_only=False)

    required_keys = {"vision_model", "classifier", "classes", "cls2idx", "backbone",
                     "hidden_size", "img_size", "best_f1", "normalization"}
    missing = required_keys - set(ckpt.keys())
    if missing:
        all_passed = failed(f"Missing checkpoint keys: {missing}")
    else:
        passed("All required keys present")

    classes = ckpt["classes"]
    cls2idx = ckpt["cls2idx"]
    num_classes = len(classes)
    print(f"    Classes ({num_classes}): {classes}")
    print(f"    Best F1: {ckpt['best_f1']:.4f}")
    print(f"    Backbone: {ckpt['backbone']}")
    print(f"    img_size: {ckpt['img_size']}")
    print(f"    hidden_size: {ckpt['hidden_size']}")

    if ckpt["img_size"] != IMG_SIZE:
        all_passed = failed(f"img_size mismatch: ckpt={ckpt['img_size']} vs expected={IMG_SIZE}")
    else:
        passed(f"img_size matches: {IMG_SIZE}")

    if ckpt["hidden_size"] != HIDDEN_SIZE:
        all_passed = failed(f"hidden_size mismatch: ckpt={ckpt['hidden_size']} vs expected={HIDDEN_SIZE}")
    else:
        passed(f"hidden_size matches: {HIDDEN_SIZE}")

    # Check classifier weight shape
    clf_weight_key = "2.weight"  # Sequential index: LayerNorm(0), Dropout(1), Linear(2)
    if clf_weight_key in ckpt["classifier"]:
        shape = ckpt["classifier"][clf_weight_key].shape
        expected = (num_classes, HIDDEN_SIZE)
        if shape != expected:
            all_passed = failed(f"Classifier weight shape {shape} != expected {expected}")
        else:
            passed(f"Classifier weight shape: {shape}")
    else:
        warn("Could not find classifier linear weight for shape check")

    norm = ckpt.get("normalization", {})
    print(f"    Normalization: mean={norm.get('mean')}, std={norm.get('std')}")

    # ──────────────────────────────────────────────
    # 2. LOAD MODEL + PYTORCH SANITY CHECK
    # ──────────────────────────────────────────────
    print(f"\n{'─'*50}")
    print(f"  [2/10] PyTorch Model Loading & Sanity Check")
    print(f"{'─'*50}")

    model = SigLIP2Classifier(SIGLIP2_MODEL_ID, num_classes, dropout_rate=0.0).to(DEVICE)
    model.vision.load_state_dict(ckpt["vision_model"])
    model.classifier.load_state_dict(ckpt["classifier"])
    model.eval()
    passed("Model loaded successfully")

    # Random input sanity check
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)

    with torch.no_grad():
        logits = model(dummy)

    if logits.shape != (1, num_classes):
        all_passed = failed(f"Logits shape {logits.shape} != expected (1, {num_classes})")
    else:
        passed(f"Logits shape: {logits.shape}")

    probs_pt = F.softmax(logits, dim=1)
    prob_sum = probs_pt.sum().item()
    if abs(prob_sum - 1.0) > 1e-4:
        all_passed = failed(f"Softmax sum = {prob_sum:.6f} (expected 1.0)")
    else:
        passed(f"Softmax sum: {prob_sum:.6f}")

    print(f"    Sample probs: {probs_pt.cpu().numpy().round(4)}")

    # ──────────────────────────────────────────────
    # 3. ONNX EXPORT
    # ──────────────────────────────────────────────
    print(f"\n{'─'*50}")
    print(f"  [3/10] ONNX Export")
    print(f"{'─'*50}")

    export_model = SigLIP2ForExport(model).eval().to(DEVICE)

    # Verify export wrapper
    with torch.no_grad():
        export_out = export_model(dummy)
    if not torch.allclose(export_out, probs_pt, atol=1e-6):
        all_passed = failed("Export wrapper output differs from manual softmax!")
    else:
        passed("Export wrapper matches manual softmax")

    onnx_path = onnx_dir / f"{onnx_name}.onnx"
    print(f"    Exporting to: {onnx_path}")

    t0 = time.time()
    with torch.no_grad():
        torch.onnx.export(
            export_model,
            dummy,
            str(onnx_path),
            opset_version=18,
            input_names=["pixel_values"],
            output_names=["probabilities"],
            dynamic_axes=None,  # fixed batch=1 for TRT
        )
    export_time = time.time() - t0

    if not onnx_path.exists():
        return failed("ONNX file not created!")

    file_size_mb = os.path.getsize(onnx_path) / 1e6
    passed(f"ONNX exported: {file_size_mb:.0f} MB in {export_time:.1f}s")

    # ──────────────────────────────────────────────
    # 4. ONNX STRUCTURE VERIFICATION
    # ──────────────────────────────────────────────
    print(f"\n{'─'*50}")
    print(f"  [4/10] ONNX Structure Verification")
    print(f"{'─'*50}")

    onnx_model = onnx.load(str(onnx_path))

    try:
        onnx.checker.check_model(onnx_model)
        passed("ONNX checker passed")
    except Exception as e:
        all_passed = failed(f"ONNX checker failed: {e}")

    # Check input
    inp = onnx_model.graph.input[0]
    inp_name = inp.name
    inp_shape = [d.dim_value for d in inp.type.tensor_type.shape.dim]
    print(f"    Input:  name='{inp_name}', shape={inp_shape}")

    if inp_name != "pixel_values":
        all_passed = failed(f"Input name '{inp_name}' != expected 'pixel_values'")
    else:
        passed("Input name: pixel_values")

    if inp_shape != [1, 3, IMG_SIZE, IMG_SIZE]:
        all_passed = failed(f"Input shape {inp_shape} != expected [1, 3, {IMG_SIZE}, {IMG_SIZE}]")
    else:
        passed(f"Input shape: {inp_shape}")

    # Check output
    out = onnx_model.graph.output[0]
    out_name = out.name
    out_shape = [d.dim_value for d in out.type.tensor_type.shape.dim]
    print(f"    Output: name='{out_name}', shape={out_shape}")

    if out_name != "probabilities":
        all_passed = failed(f"Output name '{out_name}' != expected 'probabilities'")
    else:
        passed("Output name: probabilities")

    if out_shape != [1, num_classes]:
        all_passed = failed(f"Output shape {out_shape} != expected [1, {num_classes}]")
    else:
        passed(f"Output shape: [1, {num_classes}]")

    del onnx_model  # free memory

    # ──────────────────────────────────────────────
    # 5. ONNX RUNTIME NUMERICAL VALIDATION
    # ──────────────────────────────────────────────
    print(f"\n{'─'*50}")
    print(f"  [5/10] ONNX Runtime Numerical Validation")
    print(f"{'─'*50}")

    # Use CPU provider for validation (works on all setups)
    providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
    try:
        sess = ort.InferenceSession(str(onnx_path), providers=providers)
        active_provider = sess.get_providers()[0]
        passed(f"ORT session created ({active_provider})")
    except Exception as e:
        all_passed = failed(f"ORT session failed: {e}")
        sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
        passed("Fallback to CPU provider")

    # Test with multiple random inputs
    n_test = 5
    max_diff_all = 0.0
    print(f"    Testing {n_test} random inputs...")

    for i in range(n_test):
        test_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)

        # PyTorch
        with torch.no_grad():
            pt_out = export_model(test_input).cpu().numpy()

        # ONNX
        ort_out = sess.run(None, {"pixel_values": test_input.cpu().numpy()})[0]

        max_diff = np.max(np.abs(pt_out - ort_out))
        max_diff_all = max(max_diff_all, max_diff)

        # Check probabilities sum to 1
        ort_sum = ort_out.sum()
        if abs(ort_sum - 1.0) > 1e-3:
            all_passed = failed(f"  Test {i+1}: ORT prob sum = {ort_sum:.6f}")

        if not np.allclose(pt_out, ort_out, atol=ATOL, rtol=RTOL):
            all_passed = failed(f"  Test {i+1}: max_diff={max_diff:.6f} exceeds tolerance")

    if max_diff_all < ATOL:
        passed(f"All {n_test} tests passed — max diff: {max_diff_all:.8f}")
    else:
        warn(f"Max diff across tests: {max_diff_all:.8f} (tolerance: atol={ATOL})")

    # ──────────────────────────────────────────────
    # 6. OUTPUT TENSOR DEEP CHECK
    # ──────────────────────────────────────────────
    print(f"\n{'─'*50}")
    print(f"  [6/10] Output Tensor Deep Check")
    print(f"{'─'*50}")

    # Use a deterministic input (all 0.5 = gray image in SigLIP2 normalization)
    gray_input = np.full((1, 3, IMG_SIZE, IMG_SIZE), 0.0, dtype=np.float32)  # normalized gray
    ort_gray = sess.run(None, {"pixel_values": gray_input})[0]

    # All probabilities should be positive
    if np.all(ort_gray >= 0):
        passed("All probabilities >= 0")
    else:
        all_passed = failed(f"Negative probabilities found: min={ort_gray.min():.6f}")

    # All probabilities should be <= 1
    if np.all(ort_gray <= 1.0 + 1e-6):
        passed("All probabilities <= 1.0")
    else:
        all_passed = failed(f"Probability > 1.0 found: max={ort_gray.max():.6f}")

    # Sum should be ~1.0
    gray_sum = ort_gray.sum()
    if abs(gray_sum - 1.0) < 1e-3:
        passed(f"Probability sum: {gray_sum:.6f}")
    else:
        all_passed = failed(f"Probability sum: {gray_sum:.6f} (expected 1.0)")

    # No NaN or Inf
    if not np.any(np.isnan(ort_gray)) and not np.any(np.isinf(ort_gray)):
        passed("No NaN/Inf values")
    else:
        all_passed = failed("NaN or Inf detected in output!")

    # Check no single class dominates suspiciously (all same value = double-softmax bug)
    ort_max = ort_gray.max()
    ort_min = ort_gray.min()
    ort_range = ort_max - ort_min
    print(f"    Gray image probs: min={ort_min:.4f}, max={ort_max:.4f}, range={ort_range:.4f}")
    print(f"    Per-class: {dict(zip(classes, ort_gray[0].round(4)))}")

    if ort_range < 0.01:
        warn("Very uniform output on gray image — possible double-softmax issue")
        warn("But this is on a blank input, so may be expected")

    # ──────────────────────────────────────────────
    # 7. REAL IMAGE VALIDATION (if available)
    # ──────────────────────────────────────────────
    print(f"\n{'─'*50}")
    print(f"  [7/10] Real Image Validation")
    print(f"{'─'*50}")

    test_dir = config.get("test_images_dir")
    if test_dir and Path(test_dir).exists():
        img_files = [f for f in Path(test_dir).iterdir()
                     if f.suffix.lower() in {'.jpg', '.jpeg', '.png'}][:5]

        if img_files:
            print(f"    Testing {len(img_files)} real images from {test_dir}")
            for img_path in img_files:
                inp_np = preprocess_image(str(img_path))

                # PyTorch
                inp_pt = torch.tensor(inp_np, dtype=torch.float32, device=DEVICE)
                with torch.no_grad():
                    pt_probs = export_model(inp_pt).cpu().numpy()

                # ONNX
                ort_probs = sess.run(None, {"pixel_values": inp_np.astype(np.float32)})[0]

                max_diff = np.max(np.abs(pt_probs - ort_probs))
                pt_pred = classes[pt_probs.argmax()]
                ort_pred = classes[ort_probs.argmax()]
                ort_conf = ort_probs.max()

                match = "✅" if pt_pred == ort_pred else "❌"
                print(f"    {match} {img_path.name[:35]:35s}  PT={pt_pred:15s}  ORT={ort_pred:15s}  "
                      f"conf={ort_conf:.3f}  diff={max_diff:.6f}")

                if pt_pred != ort_pred:
                    all_passed = failed(f"Prediction mismatch on {img_path.name}")
            passed("Real image validation complete")
        else:
            warn("No images found in test directory")
    else:
        warn("No test_images_dir set — skipping real image validation")

    # ──────────────────────────────────────────────
    # 8. ONNX SIMPLIFICATION
    # ──────────────────────────────────────────────
    print(f"\n{'─'*50}")
    print(f"  [8/10] ONNX Simplification (onnxsim)")
    print(f"{'─'*50}")

    sim_path = onnx_dir / f"{onnx_name}_sim.onnx"

    try:
        import onnxsim
        onnx_model = onnx.load(str(onnx_path))
        sim_model, check = onnxsim.simplify(onnx_model)

        if check:
            onnx.save(sim_model, str(sim_path))
            sim_size = os.path.getsize(sim_path) / 1e6
            orig_size = os.path.getsize(onnx_path) / 1e6
            reduction = (1 - sim_size / orig_size) * 100
            passed(f"Simplified: {orig_size:.0f}MB → {sim_size:.0f}MB ({reduction:.1f}% reduction)")
        else:
            all_passed = failed("onnxsim simplification check failed")
            sim_path = onnx_path  # fallback to original

        del onnx_model, sim_model
    except ImportError:
        warn("onnxsim not installed — skipping simplification")
        warn("Install with: pip install onnxsim")
        sim_path = onnx_path
    except Exception as e:
        warn(f"Simplification failed: {e}")
        sim_path = onnx_path

    # ──────────────────────────────────────────────
    # 9. SIMPLIFIED ONNX VALIDATION
    # ──────────────────────────────────────────────
    print(f"\n{'─'*50}")
    print(f"  [9/10] Simplified ONNX Validation")
    print(f"{'─'*50}")

    if sim_path.exists() and sim_path != onnx_path:
        # Validate simplified model
        sim_onnx = onnx.load(str(sim_path))
        try:
            onnx.checker.check_model(sim_onnx)
            passed("Simplified ONNX checker passed")
        except Exception as e:
            all_passed = failed(f"Simplified ONNX checker failed: {e}")

        # Check I/O names preserved
        sim_inp_name = sim_onnx.graph.input[0].name
        sim_out_name = sim_onnx.graph.output[0].name
        if sim_inp_name == "pixel_values" and sim_out_name == "probabilities":
            passed("I/O names preserved after simplification")
        else:
            all_passed = failed(f"I/O names changed: in='{sim_inp_name}', out='{sim_out_name}'")

        del sim_onnx

        # Numerical check on simplified model
        sess_sim = ort.InferenceSession(str(sim_path), providers=providers)
        test_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)

        with torch.no_grad():
            pt_out = export_model(test_input).cpu().numpy()
        sim_out = sess_sim.run(None, {"pixel_values": test_input.cpu().numpy()})[0]

        max_diff = np.max(np.abs(pt_out - sim_out))
        if max_diff < ATOL:
            passed(f"Simplified model matches PyTorch (max_diff={max_diff:.8f})")
        else:
            all_passed = failed(f"Simplified model differs: max_diff={max_diff:.6f}")

        del sess_sim
    else:
        warn("No simplified model to validate")

    # ──────────────────────────────────────────────
    # 10. PRODUCTION CONFIG EXPORT
    # ──────────────────────────────────────────────
    print(f"\n{'─'*50}")
    print(f"  [10/10] Production Config Export")
    print(f"{'─'*50}")

    # Load thresholds
    thresh_path = config.get("thresh_path")
    thresholds = {}
    if thresh_path and Path(thresh_path).exists():
        with open(thresh_path) as f:
            thresh_data = json.load(f)
        thresholds = thresh_data.get("thresholds", {})
        passed(f"Loaded {len(thresholds)} thresholds")
    else:
        warn("No threshold file — using 0.5 defaults")
        thresholds = {c: 0.5 for c in classes}

    prod_config = {
        "model": f"SigLIP2_SO400M_{branch.title()}",
        "model_id": SIGLIP2_MODEL_ID,
        "branch": branch,
        "img_size": IMG_SIZE,
        "num_classes": num_classes,
        "class_names": classes,
        "class_to_idx": cls2idx,
        "thresholds": thresholds,
        "normalization": {
            "mean": list(NORM_MEAN),
            "std": list(NORM_STD),
        },
        "preprocessing": {
            "normalize_mean": list(NORM_MEAN),
            "normalize_std": list(NORM_STD),
            "resize": IMG_SIZE,
            "interpolation": "bilinear",
        },
        "onnx": {
            "file": sim_path.name if sim_path.exists() else onnx_path.name,
            "input_name": "pixel_values",
            "input_shape": [1, 3, IMG_SIZE, IMG_SIZE],
            "output_name": "probabilities",
            "output_shape": [1, num_classes],
            "output_includes_softmax": True,
        },
        "training_info": {
            "best_f1": float(ckpt["best_f1"]),
            "backbone": ckpt["backbone"],
            "hidden_size": HIDDEN_SIZE,
            "epoch": ckpt.get("epoch", "N/A"),
        },
        "trt_conversion": {
            "fp16_command": f"trtexec --onnx={sim_path.name} --saveEngine={onnx_name}_fp16.engine --fp16 --memPoolSize=workspace:8192M",
            "fp32_command": f"trtexec --onnx={sim_path.name} --saveEngine={onnx_name}_fp32.engine --memPoolSize=workspace:8192M",
            "note": "Use FP16 first. If attention layers lose precision, fall back to FP32.",
        },
        "production_notes": [
            "Output 'probabilities' already has softmax applied",
            "Do NOT apply softmax again — causes double-softmax bug (all outputs ~0.475)",
            "For temperature scaling: logits = log(clip(probs, 1e-7, 1.0)); logits /= T; probs = softmax(logits)",
        ],
    }

    config_path = onnx_dir / f"{onnx_name}_production_config.json"
    with open(config_path, "w") as f:
        json.dump(prod_config, f, indent=2)
    passed(f"Config saved: {config_path}")

    # ──────────────────────────────────────────────
    # SUMMARY
    # ──────────────────────────────────────────────
    print(f"\n{'='*70}")
    if all_passed:
        print(f"  ✅ ALL VALIDATIONS PASSED — {branch.upper()} model ready for TRT conversion")
    else:
        print(f"  ⚠️  SOME VALIDATIONS FAILED — review above before deploying")
    print(f"{'='*70}")

    print(f"\n  Files created:")
    print(f"    ONNX (original):    {onnx_path}")
    if sim_path != onnx_path:
        print(f"    ONNX (simplified):  {sim_path}  ← USE THIS FOR TRT")
    print(f"    Production config:  {config_path}")

    print(f"\n  TRT conversion commands (run on Jetson AGX Orin):")
    print(f"    # Try FP16 first (faster inference):")
    print(f"    trtexec --onnx={sim_path.name} --saveEngine={onnx_name}_fp16.engine --fp16 --memPoolSize=workspace:8192M")
    print(f"\n    # If FP16 has precision issues, use FP32:")
    print(f"    trtexec --onnx={sim_path.name} --saveEngine={onnx_name}_fp32.engine --memPoolSize=workspace:8192M")

    # Cleanup
    del model, export_model, sess
    torch.cuda.empty_cache()

    return all_passed


# ============================================================
# RUN BOTH
# ============================================================
print(f"{'='*70}")
print(f"  SigLIP2 Sub-Classifier — PT → ONNX Conversion Pipeline")
print(f"  Device: {DEVICE}")
print(f"{'='*70}")

results = {}
for branch, config in MODELS.items():
    results[branch] = convert_and_validate(branch, config)

# Final summary
print(f"\n\n{'='*70}")
print(f"  FINAL SUMMARY")
print(f"{'='*70}")
for branch, passed_all in results.items():
    status = "✅ READY" if passed_all else "⚠️  NEEDS REVIEW"
    print(f"  {branch.upper():>8s}: {status}")

print(f"\n  Next steps:")
print(f"    1. Copy *_sim.onnx + *_production_config.json to Jetson AGX Orin")
print(f"    2. Run trtexec to build .engine files")
print(f"    3. Update inf_macro.py to load new SigLIP2 sub-classifier engines")
print(f"    4. Production code must use normalization mean/std = [0.5, 0.5, 0.5]")
print(f"       (different from ConvNeXt's ImageNet normalization!)")
print(f"{'='*70}")

In [ ]:
import os, json, sys, warnings, time
from pathlib import Path
import numpy as np
from PIL import Image

import torch
import torch.nn.functional as F
from torch import nn

import onnx
# onnxruntime and onnxsim are typically installed in the notebook where the original code was run
# import onnxruntime as ort
# import onnxsim

warnings.filterwarnings("ignore")

### Configuration

### Configuration
Update the `MODELS` dictionary with the correct paths for your specific model (e.g., 'upper' or 'lower' sub-classifier).

In [ ]:
MODELS = {
    "lower": {
        "pt_path": "/content/drive/Shareddrives/Garment Type/Complete_dataset/Siglip2_models/siglip2_lower_sub/best_siglip2_lower.pt",
        "onnx_dir": "/content/drive/Shareddrives/Garment Type/Complete_dataset/Siglip2_models/siglip2_lower_sub",
        "onnx_name": "siglip2_lower",
        "thresh_path": "/content/drive/Shareddrives/Garment Type/Complete_dataset/Siglip2_models/siglip2_lower_sub/lower_siglip2_thresholds.json",
        "test_images_dir": None,
    },
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SIGLIP2_MODEL_ID = "google/siglip2-so400m-patch14-384"
IMG_SIZE = 384
HIDDEN_SIZE = 1152
NORM_MEAN = (0.5, 0.5, 0.5)
NORM_STD = (0.5, 0.5, 0.5)

### Model Definitions

In [ ]:
class SigLIP2Classifier(nn.Module):
    """Must match training definition exactly."""
    def __init__(self, model_id, num_classes, dropout_rate=0.0):
        super().__init__()
        from transformers import AutoModel
        full_model = AutoModel.from_pretrained(model_id, ignore_mismatched_sizes=True)
        self.vision = full_model.vision_model
        del full_model

        self.classifier = nn.Sequential(
            nn.LayerNorm(HIDDEN_SIZE),
            nn.Dropout(dropout_rate),
            nn.Linear(HIDDEN_SIZE, num_classes),
        )

    def forward(self, pixel_values):
        # For ONNX export, we need to provide all expected inputs to the vision model.
        # Based on SIGLIP2_MODEL_ID = "google/siglip2-so400m-patch14-384"
        # Patch size is 14, Image size is 384. 384 // 14 = 27.
        # This suggests a patch grid of 27x27 = 729 patches.
        batch_size = pixel_values.shape[0]
        num_patches_per_dim = IMG_SIZE // 14 # Assuming 14 is the patch size
        num_patches_flat = num_patches_per_dim * num_patches_per_dim # 27 * 27 = 729

        # Create dummy pixel_attention_mask (all ones for full image)
        pixel_attention_mask = torch.ones(
            (batch_size, num_patches_flat),
            dtype=torch.long, device=pixel_values.device
        )

        # Create dummy spatial_shapes
        spatial_shapes = torch.tensor(
            [[num_patches_per_dim, num_patches_per_dim]],
            dtype=torch.long, device=pixel_values.device
        )

        outputs = self.vision(
            pixel_values=pixel_values,
            pixel_attention_mask=pixel_attention_mask,
            spatial_shapes=spatial_shapes
        )
        pooled = outputs.pooler_output
        return self.classifier(pooled)


class SigLIP2ForExport(nn.Module):
    """Wrapper: bakes softmax into output for TRT deployment.
    Output tensor is 'probabilities' — do NOT apply softmax again."""
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, pixel_values):
        logits = self.model(pixel_values)
        return F.softmax(logits, dim=1)

### ONNX Conversion

In [ ]:
def convert_and_export_model(branch, config):
    print(f"\n{'#'*70}")
    print(f"  CONVERTING: SigLIP2 {branch.upper()} Sub-Classifier")
    print(f"{'#'*70}")

    pt_path = Path(config["pt_path"])
    onnx_dir = Path(config["onnx_dir"])
    onnx_name = config["onnx_name"]
    onnx_dir.mkdir(parents=True, exist_ok=True)

    if not pt_path.exists():
        print(f"❌ Checkpoint not found: {pt_path}")
        return False

    print(f"Loading PyTorch checkpoint from: {pt_path}")
    ckpt = torch.load(pt_path, map_location="cpu", weights_only=False)

    classes = ckpt["classes"]
    cls2idx = ckpt["cls2idx"]
    num_classes = len(classes)
    print(f"  Classes ({num_classes}): {classes}")

    print(f"Loading SigLIP2Classifier model...")
    model = SigLIP2Classifier(SIGLIP2_MODEL_ID, num_classes, dropout_rate=0.0).to(DEVICE)
    model.vision.load_state_dict(ckpt["vision_model"])
    model.classifier.load_state_dict(ckpt["classifier"])
    model.eval()
    print("  ✅ PyTorch model loaded successfully")

    export_model = SigLIP2ForExport(model).eval().to(DEVICE)

    dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    onnx_path = onnx_dir / f"{onnx_name}.onnx"

    print(f"\nExporting ONNX model to: {onnx_path}")
    with torch.no_grad():
        torch.onnx.export(
            export_model,
            dummy_input,
            str(onnx_path),
            opset_version=18,
            input_names=["pixel_values"],
            output_names=["probabilities"],
            dynamic_axes=None,
        )
    print(f"  ✅ ONNX model exported. File size: {os.path.getsize(onnx_path) / 1e6:.0f} MB")

    # Verify ONNX model
    try:
        onnx_model = onnx.load(str(onnx_path))
        onnx.checker.check_model(onnx_model)
        print("  ✅ ONNX model verification passed (onnx.checker)")
    except Exception as e:
        print(f"  ❌ ONNX model verification failed: {e}")
        return False

    sim_path = onnx_path # Default to original if simplification fails or not available
    try:
        import onnxsim
        print("\nAttempting ONNX simplification...")
        sim_model, check = onnxsim.simplify(onnx.load(str(onnx_path)))
        if check:
            sim_path = onnx_dir / f"{onnx_name}_sim.onnx"
            onnx.save(sim_model, str(sim_path))
            orig_size = os.path.getsize(onnx_path) / 1e6
            sim_size = os.path.getsize(sim_path) / 1e6
            reduction = (1 - sim_size / orig_size) * 100
            print(f"  ✅ ONNX model simplified: {orig_size:.0f}MB -> {sim_size:.0f}MB ({reduction:.1f}% reduction)")
        else:
            print("  ⚠️  ONNX simplification check failed.")
    except ImportError:
        print("  ⚠️  onnxsim not installed. Skipping simplification. Install with: `pip install onnxsim`")
    except Exception as e:
        print(f"  ❌ ONNX simplification failed: {e}")

    # Save production config
    thresh_path = config.get("thresh_path")
    thresholds = {}
    if thresh_path and Path(thresh_path).exists():
        with open(thresh_path) as f:
            thresh_data = json.load(f)
        thresholds = thresh_data.get("thresholds", {})
        print(f"  Loaded {len(thresholds)} thresholds from {thresh_path}")
    else:
        print("  ⚠️  No threshold file found, using default 0.5 for all classes.")
        thresholds = {c: 0.5 for c in classes}

    prod_config = {
        "model": f"SigLIP2_SO400M_{branch.title()}",
        "model_id": SIGLIP2_MODEL_ID,
        "branch": branch,
        "img_size": IMG_SIZE,
        "num_classes": num_classes,
        "class_names": classes,
        "class_to_idx": cls2idx,
        "thresholds": thresholds,
        "normalization": {
            "mean": list(NORM_MEAN),
            "std": list(NORM_STD),
        },
        "onnx": {
            "file": sim_path.name,
            "input_name": "pixel_values",
            "input_shape": [1, 3, IMG_SIZE, IMG_SIZE],
            "output_name": "probabilities",
            "output_shape": [1, num_classes],
            "output_includes_softmax": True,
        },
        "training_info": {
            "best_f1": float(ckpt["best_f1"]),
            "backbone": ckpt["backbone"],
            "hidden_size": HIDDEN_SIZE,
            "epoch": ckpt.get("epoch", "N/A"),
        },
        "trt_conversion": {
            "fp16_command": f"trtexec --onnx={sim_path.name} --saveEngine={onnx_name}_fp16.engine --fp16 --memPoolSize=workspace:8192M",
            "fp32_command": f"trtexec --onnx={sim_path.name} --saveEngine={onnx_name}_fp32.engine --memPoolSize=workspace:8192M",
            "note": "Use FP16 first. If attention layers lose precision, fall back to FP32.",
        },
        "production_notes": [
            "Output 'probabilities' already has softmax applied",
            "Do NOT apply softmax again — causes double-softmax bug (all outputs ~0.475)",
        ],
    }

    config_path = onnx_dir / f"{onnx_name}_production_config.json"
    with open(config_path, "w") as f:
        json.dump(prod_config, f, indent=2)
    print(f"  ✅ Production config saved: {config_path}")
    print(f"\n  ONNX model for TRT: {sim_path}")
    print(f"  TRT FP16 Command: {prod_config['trt_conversion']['fp16_command']}")
    print(f"  TRT FP32 Command: {prod_config['trt_conversion']['fp32_command']}")
    return True

### Run Conversion

In [ ]:
for branch, config in MODELS.items():
    convert_and_export_model(branch, config)

print("\nConversion process completed.")

In [ ]:
!pip install onnx onnxscript

In [ ]:
!pip install onnx onnxscript